In [1]:
using JuMP, Ipopt, LinearAlgebra

In [2]:
# Número de barras
nb = 3

# Matriz de susceptância (Bbus simplificada)
B = [
     10.0  -5.0  -5.0;
     -5.0  10.0  -5.0;
     -5.0  -5.0  10.0
]

# Demanda (Pd)
Pd = [0.0, 1.0, 0.5]

# Limites de geração
Pg_min = [0.0, 0.0, 0.0]
Pg_max = [2.0, 0.0, 2.0]

# Custo quadrático: a*Pg^2 + b*Pg
a = [0.1, 0.0, 0.2]
b = [1.0, 0.0, 1.2]

3-element Vector{Float64}:
 1.0
 0.0
 1.2

In [3]:
model = Model(Ipopt.Optimizer)

A JuMP Model
├ solver: Ipopt
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

In [4]:
# Geração
@variable(model, Pg_min[i] <= Pg[i=1:nb] <= Pg_max[i])

# Ângulos de tensão
@variable(model, θ[1:nb])

3-element Vector{VariableRef}:
 θ[1]
 θ[2]
 θ[3]

In [5]:
@constraint(model, θ[1] == 0)

θ[1] == 0

In [6]:
@objective(model, Min, sum(a[i]*Pg[i]^2 + b[i]*Pg[i] for i in 1:nb))

0.1 Pg[1]² + 0.2 Pg[3]² + Pg[1] + 1.2 Pg[3]

In [7]:
@constraint(model, [i=1:nb],
    Pg[i] - Pd[i] == sum(B[i,j]*(θ[i] - θ[j]) for j in 1:nb)
)

3-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.EqualTo{Float64}}, ScalarShape}}:
 Pg[1] + 10 θ[1] - 5 θ[2] - 5 θ[3] == 0
 Pg[2] - 5 θ[1] + 10 θ[2] - 5 θ[3] == 1
 Pg[3] - 5 θ[1] - 5 θ[2] + 10 θ[3] == 0.5

In [8]:
optimize!(model)


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.2.

Number of nonzeros in equality constraint Jacobian...:       12
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        2

Total number of variables............................:        5
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        2
                     variables with only upper bounds:        0
Total number of equality constraints.................:        4
Total number of inequality c

In [9]:
println("Status: ", termination_status(model))

println("\nGeração ótima:")
for i in 1:nb
    println("Pg[$i] = ", value(Pg[i]))
end

println("\nÂngulos de tensão:")
for i in 1:nb
    println("θ[$i] = ", value(θ[i]))
end

println("\nCusto mínimo:")
println(objective_value(model))

Status: LOCALLY_SOLVED

Geração ótima:
Pg[1] = 1.3333332812867333
Pg[2] = 0.0
Pg[3] = 0.1666667187132666

Ângulos de tensão:
θ[1] = 0.0
θ[2] = 0.15555555208578223
θ[3] = 0.11111110417156445

Custo mínimo:
1.7166666666666675
